# Sequence Stability Experiments

Ce notebook reprend le script `sequence_stability_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Verifie que le modele sequence live reste stable sur plusieurs splits/seeds.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated split stability for shortlisted sequence danger models.
- Commande de reproduction referencee : sequence repeated split stability, sequence extra stability repeats.
- Artefacts controles : 10-seed sequence stability exists. (`runs/exp_015_sequence_stability_10seed_shortlist/metrics/stability_summary.csv`); Additional 10-seed sequence stability extension exists. (`runs/exp_021_sequence_stability_extra10/metrics/stability_summary.csv`); Combined 20-seed sequence stability summary exists. (`runs/exp_022_sequence_stability_20seed_combined/stability_summary_20seed.md`).
- Run par defaut : `runs/exp_012_sequence_stability_shortlist`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_stability_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from ml_pipeline import ROOT, RUNS_DIR, HORIZONS, write_json
from sequence_experiments import (
    append_report,
    evaluate_catalogue_model,
    make_run_dir,
    train_one_model,
)


## Fonction `stratified_video_split`

Cette cellule definit `stratified_video_split`. Elle prepare une partie du script.

In [ ]:
def stratified_video_split(meta, seed):
    video_rows = meta.groupby("video_id", as_index=False)["is_danger_clip"].max()
    video_ids = video_rows["video_id"].to_numpy()
    labels = video_rows["is_danger_clip"].astype(int).to_numpy()
    train_ids, temp_ids, _, temp_labels = train_test_split(
        video_ids,
        labels,
        test_size=0.30,
        random_state=seed,
        stratify=labels,
    )
    val_ids, test_ids = train_test_split(
        temp_ids,
        test_size=0.50,
        random_state=seed + 1,
        stratify=temp_labels,
    )
    split = {video_id: "train" for video_id in train_ids}
    split.update({video_id: "val" for video_id in val_ids})
    split.update({video_id: "test" for video_id in test_ids})
    return split


## Fonction `normalize_for_split`

Cette cellule definit `normalize_for_split`. Elle prepare une partie du script.

In [ ]:
def normalize_for_split(X_raw, meta):
    train_mask = meta["split"].to_numpy() == "train"
    flat = X_raw[train_mask].reshape(-1, X_raw.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0)
    std = np.where(std > 1e-6, std, 1.0)
    return ((X_raw - mean) / std).astype(np.float32), mean, std


## Fonction `selection_score`

Cette cellule definit `selection_score`. Elle prepare une partie du script.

In [ ]:
def selection_score(row):
    return (
        float(row.get("average_precision", 0) or 0)
        + 0.5 * float(row.get("best_hit_rate", 0) or 0)
        + 0.2 * float(row.get("best_window_precision", 0) or 0)
        - 0.03 * min(float(row.get("best_false_alarms_per_min", 20) or 20), 20.0)
    )


## Fonction `run_stability`

Cette cellule definit `run_stability`. Elle prepare une partie du script.

In [ ]:
def run_stability(args):
    source_run = Path(args.sequence_run)
    if not source_run.is_absolute():
        source_run = ROOT / source_run
    run_dir = make_run_dir(args.run_name)
    data = np.load(source_run / "features" / "sequence_dataset.npz")
    X_norm = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    base_meta = pd.read_csv(source_run / "features" / "sequence_index.csv")
    seq_config = {
        "sequence_run": str(source_run),
        "seeds": args.seeds,
        "epochs": args.epochs,
        "patience": args.patience,
        "batch_size": args.batch_size,
        "note": "Repeated video-level splits; each split re-normalized using its own training videos.",
    }
    write_json(run_dir / "config.json", seq_config)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)

    specs = [
        ("tcn_aug_bce", "tcn", True, "bce"),
        ("tcn_noaug_bce", "tcn", False, "bce"),
        ("tcn_aug_focal", "tcn", True, "focal"),
        ("tcn_noaug_focal", "tcn", False, "focal"),
        ("cnn1d_aug_focal", "cnn1d", True, "focal"),
        ("gru_noaug_bce", "gru", False, "bce"),
        ("cnn_gru_aug_bce", "cnn_gru", True, "bce"),
        ("lstm_aug_bce", "lstm", True, "bce"),
    ]
    if args.quick:
        specs = specs[:3]

    all_metrics = []
    all_history = []
    split_rows = []
    for seed in args.seeds:
        split = stratified_video_split(base_meta, seed)
        meta = base_meta.copy()
        meta["split"] = meta["video_id"].map(split)
        X, split_mean, split_std = normalize_for_split(X_raw, meta)
        meta.to_csv(run_dir / "features" / f"split_seed_{seed}.csv", index=False)
        np.savez_compressed(run_dir / "features" / f"normalizer_seed_{seed}.npz", mean=split_mean, std=split_std)
        split_rows.append({"seed": seed, **meta.groupby("split")["video_id"].nunique().to_dict()})
        for base_name, kind, augment, loss in specs:
            model_name = f"seed{seed}_{base_name}"
            print(f"training {model_name} ({loss})")
            model_args = SimpleNamespace(
                seed=seed,
                batch_size=args.batch_size,
                lr=args.lr,
                weight_decay=args.weight_decay,
                epochs=args.epochs,
                patience=args.patience,
                loss=loss,
                label_smoothing=args.label_smoothing,
                focal_gamma=args.focal_gamma,
            )
            model, history, train_time_s, model_size_bytes = train_one_model(model_name, kind, augment, X, y, meta, run_dir, model_args, device)
            for row in history:
                row["repeat_seed"] = seed
                row["base_architecture"] = base_name
                row["loss"] = loss
            all_history.extend(history)
            rows, _ = evaluate_catalogue_model(
                model_name,
                model,
                X,
                y,
                meta,
                run_dir,
                device,
                train_time_s,
                model_size_bytes,
                args.batch_size,
                loss,
            )
            for row in rows:
                row["repeat_seed"] = seed
                row["base_architecture"] = base_name
                row["selection_score"] = selection_score(row) if row["split"] == "val" and float(row["horizon_s"]) == 1.0 else None
            all_metrics.extend(rows)
            pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "stability_architecture_comparison.csv", index=False)
            pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "stability_training_history.csv", index=False)

    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "stability_architecture_comparison.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "stability_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "stability_split_counts.csv", index=False)

    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    summary = []
    for (base_arch, split_name), group in h1.groupby(["base_architecture", "split"]):
        summary.append(
            {
                "base_architecture": base_arch,
                "split": split_name,
                "n_repeats": int(group["repeat_seed"].nunique()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "hit_rate_mean": float(group["best_hit_rate"].mean()),
                "hit_rate_std": float(group["best_hit_rate"].std(ddof=0)),
                "false_alarms_per_min_mean": float(group["best_false_alarms_per_min"].mean()),
                "false_alarms_per_min_std": float(group["best_false_alarms_per_min"].std(ddof=0)),
                "precision_mean": float(group["best_window_precision"].mean()),
            }
        )
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(run_dir / "metrics" / "stability_summary.csv", index=False)

    val_summary = summary_df[summary_df["split"] == "val"].copy()
    val_summary["stability_score"] = val_summary["ap_mean"] + 0.5 * val_summary["hit_rate_mean"] + 0.2 * val_summary["precision_mean"] - 0.03 * val_summary["false_alarms_per_min_mean"].clip(upper=20)
    val_summary = val_summary.sort_values("stability_score", ascending=False)
    lines = ["# Repeated Split Sequence Stability", ""]
    lines.append("Repeated video-level split results for the 1.0s danger horizon. Each repeat recomputes normalization from training videos only.")
    lines.append("")
    lines.append("| rank | architecture | val AP mean | val AP std | val hit mean | val FA/min mean | val precision mean |")
    lines.append("|---:|---|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(val_summary.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['base_architecture']} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | {row['hit_rate_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | {row['precision_mean']:.3f} |"
        )
    lines.append("")
    lines.append("This is still not final because it uses a shortlist, not every possible architecture/loss/augmentation combination.")
    (run_dir / "stability_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(
        run_dir,
        "Repeated Split Stability Completion",
        f"- Repeats: `{args.seeds}`\n- Specs: `{[spec[0] for spec in specs]}`\n- Summary: `{run_dir / 'stability_summary.md'}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated split stability for shortlisted sequence danger models.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_012_sequence_stability_shortlist")
    parser.add_argument("--seeds", nargs="+", type=int, default=[101, 202, 303])
    parser.add_argument("--epochs", type=int, default=25)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--quick", action="store_true")
    args = parser.parse_args()
    run_stability(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_012_sequence_stability_shortlist_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_stability_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
